In [27]:
import pandas as pd
import re
import json
import random

In [3]:
food_dir='dataset/gofood_dataset.csv'
prod_dir = 'dataset/tokopedia_product_reviews_2025.csv'
names_dir = 'dataset/Indonesian_Name_Dataset.csv'

In [4]:
df_food = pd.read_csv(food_dir)
df_food.head()

,merchant_name,merchant_area,category,display,product,price,discount_price,isDiscount,description
0,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Hot Almara Kopi (kopi Susu Gula Aren),20000.0,NaN,0,Sajian Kopi Susu Gula Aren Yang Berbeda Dari K...
1,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Ice Almara Kopi (kopi Susu Gula Aren),22000.0,NaN,0,Sajian Kopi Susu Gula Aren Yang Berbeda Dari K...
2,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Hot Millsis,20000.0,NaN,0,Sajian Susu Coklat Milo Dengan Racikan Khas 3 ...
3,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Ice Millsis,20000.0,NaN,0,Sajian Susu Coklat Milo Dengan Racikan Khas 3 ...
4,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Hot Millbro,22000.0,NaN,0,Sajian Susu Coklat Milo Plus Espresso Dengan R...


In [5]:
df_prod = pd.read_csv(prod_dir)
df_prod.head()

,review_text,review_date,review_id,product_name,product_category,product_variant,product_price,product_url,product_id,rating,sold_count,shop_id,sentiment_label
0,baru sekali ini terima brg dr belanja online d...,2024-12-22,1134256160,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
1,cocok bgt aku sama telur nya. nga Amis menurut...,2025-02-25,1242584634,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
2,Telornya sudah sampai di rumah dengan kemasan ...,2025-07-15,1573444677,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
3,Telor sudah diterima dengan baik dan tidak ada...,2025-07-20,1581728541,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
4,"Alhamdulillah penjual amanah,Telor nya terbaik...",2023-04-24,881041355,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Full Design,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive


In [6]:
df_names = pd.read_csv(names_dir)
df_names.head()

,Unnamed: 0,NAMA,Count,Kategori,Jumlah Suku Kata,jumlah kata,Jumlah Huruf Vokal,Jumlah Huruf Konsonan,Jumlah Huruf
0,0,ANJAR SUROTO,1,Pribadi,5,2,5,6,11
1,1,REINHARD RUDIYANTO SIHITE,1,Pribadi,9,3,10,13,23
2,2,ABD RAHIM NANI,1,Pribadi,5,3,5,7,12
3,3,DRS MUSTAPHA AFIFF,1,Pribadi,5,3,5,11,16
4,4,FAIZ FIRDAUS,1,Pribadi,3,2,5,6,11


In [7]:
raw_names = df_names['NAMA']
raw_names.head()

0                 ANJAR SUROTO
1    REINHARD RUDIYANTO SIHITE
2               ABD RAHIM NANI
3           DRS MUSTAPHA AFIFF
4                 FAIZ FIRDAUS
Name: NAMA, dtype: str

In [8]:
raw_names.tolist()

['ANJAR SUROTO',
 'REINHARD RUDIYANTO SIHITE',
 'ABD RAHIM NANI',
 'DRS MUSTAPHA AFIFF',
 'FAIZ FIRDAUS',
 'RIKI YULISMAN',
 'JUNARIO HP SRG. A',
 'M.ROBBY',
 'MIRHAN',
 'J.PARDEDE IV',
 'SUKARTINAH',
 'ELI SONALIA',
 'MOCH SODIKIN',
 'I KETUT WARDIANA',
 'TRINDO JUNAEDI',
 'ARI FATIMAH',
 'I DEWA GEDE WISASTA',
 'HUSAIN JAYA',
 'AHMAD SYAEFULLOH',
 'MUHAMAD KHOLIL',
 'ROCHADI C',
 'HARTON SE',
 'DANI FIQIH 06',
 'KAMRIL',
 'EKO NOVAN SAPUTRO(CUCIAN)',
 'SAMSUDIN. A',
 'SUKLI',
 'NURLELI SAGITA',
 'CHANDRA WINARSIH',
 'NI WAYAN SUDARIATI',
 'M.ZAIDI',
 'RIWOTO',
 'MIYAR / NIYAR',
 'AL HOERIAH',
 'DRS SUBAHI IDRIS MM',
 'HENDRA JAYA 02',
 'LIE ANDY KUSMANTO',
 'PETRUS APRIANTO 01',
 'HASANUDDIN S',
 'HUZEIN NOOR',
 'NURBIYANTONI',
 'DIDIET RADITYO',
 'SITI SUWARTIK',
 'ANDRE YULIUS TJAHYA S',
 'ADE MULYANI',
 'SALMAN PARSI',
 'ARI MINDAYANI',
 'LA HANUGA',
 'H. NASRUDDIN',
 'SITI HANDRIANI',
 'IIN ABDUL SALEH',
 'DAVY LENSOEN TJOMAN',
 'SETYA NINGRUM',
 'INFAN ANNUR NAGA',
 'WIBAWA SAPU

In [9]:
EXTERNAL_NAMES = []

In [10]:
for name in raw_names:
    clean_name = re.sub(r'\b(DRS|DR|IR|HJ|H|ST|SE|SPD|SH)\b', '', name, flags=re.IGNORECASE).strip()
    
    parts = clean_name.split()
    if parts:
        EXTERNAL_NAMES.append(parts[0].capitalize())

In [11]:
EXTERNAL_NAMES = list(set(EXTERNAL_NAMES))

In [12]:
raw_food = df_food['product'].dropna().tolist()
raw_food

['Hot Almara Kopi (kopi Susu Gula Aren)',
 'Ice Almara Kopi (kopi Susu Gula Aren)',
 'Hot Millsis',
 'Ice Millsis',
 'Hot Millbro',
 'Ice Millbro',
 'Hot 330 Helateh',
 'Ice 330 Helateh',
 'Hot Ilusi Kopi',
 'Ice Ilusi Kopi',
 'Kopi Jelly',
 'Hot Americano',
 'Ice Americano',
 'Hot Latte',
 'Ice Latte',
 'Hot Cappucino',
 'Ice Cappucino',
 'Hot Mocacino',
 'Ice Mocacino',
 'Hot Caramel Latte',
 'Ice Frappucino',
 'Ice Caramel Latte',
 'Hot Hazelnut Latte',
 'Ice Hazelnut Latte',
 'Hot Vanilla Latte',
 'Ice Vanilla Latte',
 'Vietnam Drip',
 'Kopi Tarik',
 'Kopi Matcha',
 'Kopi Charcoal',
 'Hot Matcha',
 'Ice Matcha',
 'Hot Charcoal',
 'Ice Charcoal',
 'Hot Coklat',
 'Ice Coklat',
 'Hot Lemon Tea',
 'Ice Lemon Tea',
 'Ice Yakult Squash',
 'Kopi Beer',
 'Teh Jelly',
 'Lemon Squash',
 'Ocean Blue',
 'Chocomint',
 'Lychee Squash',
 'Chocoreo',
 'Pisang Lumer Coklat',
 'Roti Bakar',
 'Kentang Goreng',
 'Paket Ngemil 1',
 'Paket Ngemil 2',
 'Paket Ngemil 3',
 'Paket Mix Bento',
 'Paket Bento 

In [13]:
clean_gofood = []

In [14]:
for item in raw_food:
    item = re.sub(r'\(.*?\)', '', item).strip()
    
    short_item = " ".join(item.split()[:3]).lower()
    clean_gofood.append(short_item)

In [15]:
clean_gofood

['hot almara kopi',
 'ice almara kopi',
 'hot millsis',
 'ice millsis',
 'hot millbro',
 'ice millbro',
 'hot 330 helateh',
 'ice 330 helateh',
 'hot ilusi kopi',
 'ice ilusi kopi',
 'kopi jelly',
 'hot americano',
 'ice americano',
 'hot latte',
 'ice latte',
 'hot cappucino',
 'ice cappucino',
 'hot mocacino',
 'ice mocacino',
 'hot caramel latte',
 'ice frappucino',
 'ice caramel latte',
 'hot hazelnut latte',
 'ice hazelnut latte',
 'hot vanilla latte',
 'ice vanilla latte',
 'vietnam drip',
 'kopi tarik',
 'kopi matcha',
 'kopi charcoal',
 'hot matcha',
 'ice matcha',
 'hot charcoal',
 'ice charcoal',
 'hot coklat',
 'ice coklat',
 'hot lemon tea',
 'ice lemon tea',
 'ice yakult squash',
 'kopi beer',
 'teh jelly',
 'lemon squash',
 'ocean blue',
 'chocomint',
 'lychee squash',
 'chocoreo',
 'pisang lumer coklat',
 'roti bakar',
 'kentang goreng',
 'paket ngemil 1',
 'paket ngemil 2',
 'paket ngemil 3',
 'paket mix bento',
 'paket bento shrimp',
 'paket bento ebi',
 'paket bento e

In [16]:
raw_prod = df_prod['product_name'].dropna().tolist()
clean_prod = []

In [17]:
clean_prod

[]

In [18]:
for item in raw_prod:
    item = str(item).split('-')[0].strip()
    
    short_item = " ".join(item.split()[:3]).lower()
    clean_prod.append(short_item)

EXTERNAL_ITEMS = list(set(clean_gofood + clean_prod))

In [19]:
EXTERNAL_ITEMS

['coockies and cream',
 'mlcc 220 22pf',
 'lapis',
 'sony kd85x80l',
 'intel 2 telur',
 'ice mocchacino',
 'roti bunting mac',
 'ontbijtkoek loaf',
 'mexican boy',
 'selai srikaya tk',
 'astacala ice',
 'white medium',
 'tipker coklat pisang',
 'es rumput laut',
 'seblak puyu',
 'siomay bandung',
 'xundd case magnetic',
 'french fries +',
 'selai hazelnut crispy',
 'garlic cheese',
 'rujak serut',
 'biscoff cake',
 'mie kuah spesial',
 'passion fruit tea',
 'chicken cheeseburger',
 'roti bakar silverqueen',
 'traktirdiver-mu',
 'coffee jelly',
 'aren latte',
 'cinnamon ginger lemonade',
 'fried prawn youtiao',
 'ikan sambal dabu-dabu',
 'sambal rica-rica',
 'nutrisari sweet mango',
 'mamasuka saus sambal',
 'coklat muffin',
 'cap cai polos',
 'all variant',
 'kopi ngunjuk panas',
 'donat siram strawberry',
 'istimewa campur keju',
 'isi vanila',
 'cheese burger +',
 'brownies choco peanut',
 'tiga rasa',
 'garlic beef du',
 'kepiting mix udang',
 'seafood pasta',
 'mie udang kering',
 

In [20]:
EXTERNAL_NAMES

['Warnet',
 'Iliardi',
 'Sapung',
 'Ice',
 'Andre',
 'Reni',
 'Palti',
 'Kadek',
 'Firda',
 'Kaefiyah',
 'Mahesti',
 'Siyama',
 'Amri',
 'Saka',
 'Merly',
 'W',
 'Kpu',
 'Erwan',
 'Sri',
 'Nopendi',
 'Safta',
 'Misbah',
 'Ruswanto',
 'Marsha',
 'Imah',
 'Arta',
 'Fitriya',
 'Nurzadian',
 'Salim',
 'Supiyo.sm',
 'M.azmien',
 'M.azmi',
 'Ny',
 'Siswo',
 'Ruhaeni',
 'Ima',
 'Khaeridah',
 'Anjar',
 'Risjal',
 'Gojali',
 'Albert',
 'Ria',
 'Boih',
 'Sutarman',
 'Fairuz',
 'Liana',
 'Hamdani',
 'Evi',
 'Zeha',
 'Djong',
 'Samto',
 'Amat',
 'Stasiun',
 'Yulia',
 'Djarotdjarotun',
 'Bobby',
 'Risa',
 'Sukiman',
 'Marhuma,',
 'Mamay',
 'Tasbehul',
 'Sanggar',
 'Tutik',
 'Jusup',
 'Barnoto',
 'Amirah/wariman',
 'Candra',
 'Sutarma',
 'Elaine',
 'Mulham',
 'Nurbiyantoni',
 'Ujang',
 'Yudi',
 'Tjong',
 'Mitha',
 'Wahyu',
 'Ansthelabel',
 'Edi',
 'Dybyo',
 'Yustina',
 'Gembah',
 'Djahidin',
 'Purwanta',
 'Willy',
 'Junaidi',
 'Lilis',
 'Ikn',
 'Nrfood',
 'Rismanita',
 'Ropika',
 'Elisa',
 'Chabib',

In [21]:
EXTERNAL_NAMES = [name.strip('.,/ ').capitalize() for name in EXTERNAL_NAMES if len(name) > 2 and name.lower() not in ['warnet', 'stasiun', 'kpu', 'ikn', 'bidan']]

In [22]:
print(f"Loaded {len(EXTERNAL_NAMES)} clean names and {len(EXTERNAL_ITEMS)} items!")

Loaded 1194 clean names and 23802 items!


In [24]:
def generate_random_price():
    slang = ["cepek", "goban", "gocap", "seceng", "cenggo", "goceng", "pekgo"]
    if random.random() < 0.2:
        return random.choice(slang)
    
    formats = [
        lambda: f"{random.randint(10, 500)}k",
        lambda: f"{random.randint(10, 500)} ribu",
        lambda: f"{random.randint(10, 500)}rb",
        lambda: f"{random.randint(1, 9)}.{random.randint(1,9)}jt",
        lambda: f"{random.randint(10, 999)}.000"
    ]
    return random.choice(formats)()

In [25]:
def build_massive_dataset(templates_file, output_file, multiplier=100):
    with open(templates_file, 'r', encoding='utf-8') as f:
        templates = json.load(f)
        
    augmented_data = []
    
    for template in templates:
        for _ in range(multiplier):
            new_text = template["raw_text"]
            new_entities = []
            replacements = {}
            
            for ent in template["entities"]:
                placeholder = ent["value"]
                ent_type = ent["entity"]
                
                if placeholder not in replacements:
                    if "PERSON" in placeholder:
                        replacements[placeholder] = random.choice(EXTERNAL_NAMES)
                    elif "ITEM" in placeholder:
                        replacements[placeholder] = random.choice(EXTERNAL_ITEMS)
                    elif "PRICE" in placeholder:
                        replacements[placeholder] = generate_random_price()
                    else:
                        replacements[placeholder] = placeholder
                
                new_entities.append({
                    "entity": ent_type,
                    "value": replacements[placeholder]
                })
            
            for placeholder, actual_value in replacements.items():
                new_text = new_text.replace(placeholder, str(actual_value))
                
            augmented_data.append({
                "raw_text": new_text,
                "entities": new_entities
            })
            
    random.shuffle(augmented_data)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(augmented_data, f, indent=2, ensure_ascii=False)
        
    print(f"created {len(augmented_data)} rows")

In [28]:
build_massive_dataset("talangin_synthetic_templates.json", "training_data.json", multiplier=10)

created 24900 rows
